# **$n$-gram language models**

### **Assignment 1 - Natural Language Processing - MSc. Computer Science, AUEB [2025-2026]**

> Maria Schoinaki, MSc Student
>
> Department of Informatics, Athens University of Economics and Business
>
> mar.schoinaki@aueb.gr

> Electra Papadopoulou, MSc Student
>
> Department of Informatics, Athens University of Economics and Business
>
> elec.papadopoulou@aueb.gr

> Leonidas Kontogiannis, MSc Student
>
> Department of Informatics, Athens University of Economics and Business
>
> leo.kontogiannis@aueb.gr

## Imports

In [16]:
%pip install -U nltk -q
%pip install -U scikit-learn

In [17]:
import nltk
import re
import math
import numpy as np

from nltk.corpus import brown
from collections import Counter, defaultdict
from nltk.tokenize import sent_tokenize, TweetTokenizer
from nltk.util import ngrams
from pprint import pprint
from sklearn.model_selection import train_test_split
from typing import Mapping, Sequence, Tuple, Iterable, Set, List

nltk.download('brown')
nltk.download('punkt')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## Preliminaries

In [18]:
START  = '*start*'
START1 = '*start1*'
START2 = '*start2*'
END    = '*end*'
UNK    = '*UNK*'
START_TOKENS = {START, START1, START2}

class Tools:
    @staticmethod
    def clean_sequence(sequence):
        # Remove any start tokens from the beginning of the sequence
        while sequence and sequence[0] in START_TOKENS:
            sequence.pop(0)
        # Also remove *end* token for display
        if sequence and sequence[-1] == END:
            sequence.pop()
        return ' '.join(sequence)

In [19]:
class TextPreprocessor:
    def __init__(self):
        # Initialize a TweetTokenizer instance for word-level tokenization
        TextPreprocessor.tokenizer = TweetTokenizer()

    @staticmethod
    def corpus_tokenizer(corpus):
        """
        Splits a text corpus into sentences and then tokenizes each sentence into words.
        """

        # Split the corpus into individual sentences
        sentences = sent_tokenize(corpus) # or use regex sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?|!)\s', corpus)
        # Tokenize each sentence into individual word tokens

        return [TextPreprocessor.tokenizer.tokenize(sentence) for sentence in sentences]

    @staticmethod
    def create_vocabulary(tokenized_corpus, threshold=10):
        """
        Constructs a vocabulary from a tokenized corpus by counting word frequencies
        and retaining only words that occur at least `threshold` times.
        """
        # Count all word frequencies in the corpus
        word_counts = Counter(word for sentence in tokenized_corpus for word in sentence)

        # Select words that meet or exceed the threshold
        vocabulary = {word for word, count in word_counts.items() if count >= threshold}

        return vocabulary

    @staticmethod
    def replace_with_unk(tokenized_corpus, vocabulary):
      """
      Processes the tokenized corpus by replacing words not in the given vocabulary with the '*UNK*' token.
      """
      return [[word if word in vocabulary else '*UNK*' for word in sentence] for sentence in tokenized_corpus]

## $n$-gram Language Models Implementation:

### **$n$-gram Probabilities** (`NgramLanguageModel.prob`)

Let \( h \) be the \((n-1)\)-gram **context** and \( w \) the **next token**.  
With add-\($alpha$) (Laplace) smoothing, the model estimates:

$$
\hat{P}(w \mid h) = \frac{C(h, w) + \alpha}{C(h) + \alpha |V|}
$$

where:

- \( C(h, w) \): n-gram count from the training set (`ngram_count_map[(h, w)]`)  
- \( C(h) \): (n−1)-gram context count (`n_1gram_count_map[h]`)  
- \( alpha > 0 \): smoothing hyperparameter (`alpha`)  
- \( |V| \): predictable vocabulary size (`self.V`), excluding start tokens  

**Special handling:**
- OOV tokens are mapped to `*UNK*` before scoring  
- Start tokens are never predicted; `*end*` tokens **are** predicted  

---

#### **Specific cases**

**Bigram model:**
$$
P_{\text{bigram}}(w_2 \mid w_1) =
\frac{C(w_1, w_2) + \alpha}{C(w_1) + \alpha |V|}
$$

**Trigram model:**
$$
P_{\text{trigram}}(w_3 \mid w_1, w_2) =
\frac{C(w_1, w_2, w_3) + \alpha}{C(w_1, w_2) + \alpha |V|}
$$

---

### **Sentence Log-Probability** (`sentence_log_prob`)

For a tokenized sentence ( $x_{1:m}$ ), we pad with start and end tokens and compute:

$$
\log_b P(x_{1:m}) =
\sum_{i=n}^{m'} \log_b \hat{P}(x_i \mid x_{i-n+1:i-1})
$$

where \( b = 2 \) (base-2 log).  
We **skip** probabilities for start tokens and **include** those for `*end*`.

---

### **Autocompletion**

1. **Greedy decoding** (`greedy_autocomplete`)
   $$
   \arg\max_{w \in (V \cup \{*end*\}) \setminus (\{*UNK*\} \cup \text{STARTS})}
   \hat{P}(w \mid h)
   $$
   At each step, choose the most probable next word.  
   Stop when `*end*` is reached or the maximum length is exceeded.

2. **Beam search** (`beam_search_autocomplete`)  
   Keeps the top-\( K \) most probable partial sequences (beam width = `beam_width`).  
   Expands each by one token using cumulative log-probabilities until all beams end.

---

### **Counts & Vocabulary (Training)**

Each sentence is padded as:

$$
[\underbrace{*start*, \ldots, *start*}_{n-1},\; x_1, \ldots, x_m,\; *end*]
$$

Vocabulary includes tokens with ≥10 occurrences in the training set, plus  
\(\{*end*, *UNK*, *start*, *start1*, *start2*\}\).  
Vocabulary size \(|V|\) excludes start tokens.

---

### **Why Use Log Probabilities?**

- Prevents **numerical underflow** in long sequences  
- Converts **products** into **sums** (more stable and efficient)  
- Logarithms are **monotonic**, preserving ranking for beam search  
- Cross-entropy and perplexity are **naturally defined** using logs


In [20]:
class NgramLanguageModel:
    """Add-alpha (Laplace) n-gram LM following assignment specifications exactly."""

    def __init__(self, n: int, alpha: float = 1.0):
        assert n >= 1, "n must be >= 1"
        assert alpha > 0.0, "alpha must be > 0"
        self.n = n
        self.alpha = float(alpha)

        self.vocabulary: Set[str] = set()
        self.V: int = 0  # |vocab \ START_TOKENS|
        self.ngram_count_map: Counter = Counter()
        self.n_1gram_count_map: Counter = Counter()

    def train_ngram(self, vocabulary: Iterable[str], ngram_count_map: Counter, n_1gram_count_map: Counter):
        # Build vocabulary with required special tokens
        self.vocabulary = set(vocabulary) | {END, UNK} | START_TOKENS
        self.V = len([w for w in self.vocabulary if w not in START_TOKENS])

        self.ngram_count_map = ngram_count_map
        self.n_1gram_count_map = n_1gram_count_map

    def _map_oov(self, token: str) -> str:
        return token if token in self.vocabulary else UNK

    def _normalize_context(self, context: Sequence[str]) -> Tuple[str, ...]:
        return tuple(self._map_oov(t) for t in context)

    def prob(self, token: str, context: Sequence[str]) -> float:
        """Compute P(token|context) with Laplace smoothing."""
        if len(context) != self.n - 1:
            raise ValueError(f"Context length must be {self.n - 1} for {self.n}-gram model")

        token = self._map_oov(token)
        context = self._normalize_context(context)

        ngram = context + (token,)
        ngram_count = self.ngram_count_map.get(ngram, 0)
        context_count = self.n_1gram_count_map.get(context, 0)

        # Laplace smoothing: (C(h,w) + α) / (C(h) + α * V)
        numerator = ngram_count + self.alpha
        denominator = context_count + self.alpha * self.V

        if denominator <= 0:
          return 0.0  # Should not happen with alpha > 0, but safe

        return numerator / denominator

    def log_prob(self, token: str, context: Sequence[str], base: float = 2.0) -> float:
        """Compute log probability using base 2 (for cross-entropy)."""
        p = self.prob(token, context)
        return math.log(p, base) if p > 0 else float('-inf')

    def sentence_log_prob(self, tokens: Sequence[str], base: float = 2.0) -> float:
        """Compute log probability of a sentence (sum of log probs)."""
        # Add appropriate start and end tokens
        if self.n == 1:
            sequence = list(tokens) + [END]
        elif self.n == 2:
            sequence = [START] + list(tokens) + [END]
        elif self.n == 3:
            sequence = [START1, START2] + list(tokens) + [END]
        else:
            sequence = [START] * (self.n - 1) + list(tokens) + [END]

        total_log_prob = 0.0
        for i in range(self.n - 1, len(sequence)):
            context = sequence[i - (self.n - 1):i]
            token = sequence[i]

            # Skip probabilities for start tokens (we don't predict them)
            if token in START_TOKENS:
                continue

            log_p = self.log_prob(token, context, base)
            total_log_prob += log_p

        return total_log_prob

    def fine_tune(self, tokenized_sentences, alphas=np.linspace(0.001, 1.0, 50)):
        """
        Fine-tune alpha parameter on development set (alpha ≤ 1).
        """
        best_alpha = None
        best_perplexity = float('inf')

        print(f"Fine-tuning {self.n}-gram model...")
        print(f"Testing {len(alphas)} alpha values from 0.001 to 1.0")
        print("-" * 60)

        for alpha in alphas:
            # Set alpha parameter (ensure it's <= 1)
            self.alpha = min(alpha, 1.0)

            # Evaluate on development set
            cross_entropy, perplexity = compute_cross_entropy_perplexity(self, tokenized_sentences)

            print(f"  Alpha: {alpha:.4f}, Cross Entropy: {cross_entropy:.4f}, Perplexity: {perplexity:.4f}")

            if perplexity < best_perplexity:
                best_perplexity = perplexity
                best_alpha = alpha

        print("-" * 45)
        print(f"Optimal Alpha: {best_alpha:.4f}, Best Perplexity: {best_perplexity:.4f}")
        print("-" * 45)

        # Set the best alpha
        self.alpha = best_alpha

        return best_alpha, best_perplexity

    def get_next_word_candidates(self, context: Sequence[str], top_k: int = 10) -> List[Tuple[str, float]]:
        """Get top-k candidate next words excluding UNK and start tokens."""
        context = self._normalize_context(context)

        # Get all possible candidates (vocabulary minus banned tokens)
        banned_tokens = START_TOKENS | {UNK}
        candidates = self.vocabulary - banned_tokens | {END}

        # Score each candidate
        scored_candidates = []
        for candidate in candidates:
            prob = self.prob(candidate, context)
            if prob > 0:
                scored_candidates.append((candidate, prob))

        # Sort by probability and return top-k
        scored_candidates.sort(key=lambda x: x[1], reverse=True)
        return scored_candidates[:top_k]

    def greedy_autocomplete(self, prefix_tokens: Sequence[str], max_length: int = 20) -> list:
        """
        Greedy decoding for sentence completion.
        Uses most probable next word at each step, never generates UNK.
        """
        completion = list(prefix_tokens)

        # Build initial context from prefix (last n-1 tokens)
        if len(prefix_tokens) >= self.n - 1:
            context = list(prefix_tokens[-(self.n - 1):])
        else:
            # Pad with start tokens if needed
            if self.n == 2:
                context = [START] + list(prefix_tokens)
            elif self.n == 3:
                context = [START1, START2] + list(prefix_tokens)
            else:
                context = list(prefix_tokens)
            context = context[-(self.n - 1):]  # Take last n-1 tokens

        context = self._normalize_context(context)

        for _ in range(max_length):
            # Get candidate next words (excluding UNK)
            candidates = self.get_next_word_candidates(context, top_k=1)

            if not candidates:
                break

            next_word, prob = candidates[0]

            if next_word == END:
                completion.append(END)
                break

            completion.append(next_word)

            # Update context for next prediction
            if self.n > 1:
                context = list(context) + [next_word]
                if len(context) > self.n - 1:
                    context.pop(0)

        return completion

    def beam_search_autocomplete(self, prefix_tokens: Sequence[str],
                               beam_width: int = 3, max_length: int = 20) -> list:
        """
        Beam search for more diverse completions.
        """
        # Initialize beam with prefix
        sequences = [(list(prefix_tokens), 0.0)]  # (sequence, log_prob)

        for step in range(max_length - len(prefix_tokens)):
            new_sequences = []

            for sequence, log_prob in sequences:
                # Skip if sequence ended
                if sequence and sequence[-1] == END:
                    new_sequences.append((sequence, log_prob))
                    continue

                # Build context for this sequence
                if len(sequence) >= self.n - 1:
                    context = sequence[-(self.n - 1):]
                else:
                    if self.n == 2:
                        context = [START] + sequence
                    elif self.n == 3:
                        context = [START1, START2] + sequence
                    else:
                        context = sequence
                    context = context[-(self.n - 1):]

                context = self._normalize_context(context)

                # Get candidate expansions
                candidates = self.get_next_word_candidates(context, top_k=beam_width * 2)

                for word, prob in candidates:
                    if prob <= 0:
                        continue

                    new_seq = sequence + [word]
                    new_log_prob = log_prob + math.log2(prob)
                    new_sequences.append((new_seq, new_log_prob))

            if not new_sequences:
                break

            # Keep top beam_width sequences
            new_sequences.sort(key=lambda x: x[1], reverse=True)
            sequences = new_sequences[:beam_width]

        # Return best sequence
        if not sequences:
            return prefix_tokens

        best_sequence = max(sequences, key=lambda x: x[1])[0]
        return best_sequence

### **Cross-Entropy & Perplexity** (`compute_cross_entropy_perplexity`)

The entire test corpus is treated as a continuous sequence of sentences, padded with start/end tokens:

$$
H = -\frac{1}{N} \sum_i \log_2 \hat{P}(x_i \mid h_i),
\qquad
\text{PPL} = 2^{H}
$$

where \( N \) is the total number of predicted tokens:  
- Includes all `*end*` tokens  
- Excludes all start tokens (`*start*`, `*start1*`, `*start2*`)

---


In [21]:
def compute_cross_entropy_perplexity(lm: NgramLanguageModel, corpus: Sequence[Sequence[str]]) -> Tuple[float, float]:
    """
    Compute cross-entropy and perplexity following assignment specifications exactly.
    - Count *end* tokens but not *start* tokens in total length N
    - Include P(*end*|...) but not P(*start*|...)
    - Treat entire test corpus as single sequence of sentences
    """
    total_log_prob = 0.0
    total_tokens = 0  # Counts all predicted tokens including *end*

    for sentence in corpus:
        # Add appropriate start and end tokens
        if lm.n == 1:
            sequence = list(sentence) + [END]
        elif lm.n == 2:
            sequence = [START] + list(sentence) + [END]
        elif lm.n == 3:
            sequence = [START1, START2] + list(sentence) + [END]
        else:
            sequence = [START] * (lm.n - 1) + list(sentence) + [END]

        # Process each position where we predict a token
        for i in range(lm.n - 1, len(sequence)):
            context = sequence[i - (lm.n - 1):i]
            token = sequence[i]

            # Skip probabilities for start tokens (we don't predict them)
            if token in START_TOKENS:
                continue

            log_p = lm.log_prob(token, context, base=2.0)
            total_log_prob += log_p
            total_tokens += 1  # Count this token (includes *end*)

    if total_tokens == 0:
        return float('inf'), float('inf')

    cross_entropy = -total_log_prob / total_tokens
    perplexity = 2 ** cross_entropy

    return cross_entropy, perplexity

## Data Fetching

For this assignment we used the “Brown” corpus form the NLTK library. From the available categories, we selected the “science_fiction” subset, as it provides a coherent domain of narrative text with consistent vocabulary and syntax.

In [22]:
# Load and split corpus
corpus = brown.sents(categories='science_fiction')
train_corpus, temp_corpus = train_test_split(corpus, test_size=0.2, random_state=40)
dev_corpus, test_corpus = train_test_split(temp_corpus, test_size=0.5, random_state=40)

# Preprocess data
text_processor = TextPreprocessor()
vocabulary = text_processor.create_vocabulary(train_corpus, threshold=10)

processed_train_corpus = text_processor.replace_with_unk(train_corpus, vocabulary)
processed_dev_corpus = text_processor.replace_with_unk(dev_corpus, vocabulary)
processed_test_corpus = text_processor.replace_with_unk(test_corpus, vocabulary)

## **Training, Hyperparameter Tuning, and Evaluation**

### **Building n-gram Counts**
We first construct the **$n$-gram** and **($n$−1)-gram** frequency counts from the tokenized and preprocessed training corpus using the function `build_ngram_counts()`.  
Each sentence is padded with the appropriate number of start tokens (`*start*`, `*start1*`, `*start2*`) and the `*end*` token to correctly capture sentence boundaries.  
These counts are then used to train the **bigram** and **trigram** models.

- For the **bigram model (n=2)**, we count occurrences of word pairs `(w₁, w₂)` and their unigram contexts `(w₁)`.  
- For the **trigram model (n=3)**, we count triplets `(w₁, w₂, w₃)` and their bigram contexts `(w₁, w₂)`.

This provides the necessary statistics for Laplace-smoothed probability estimation.

In [23]:
def build_ngram_counts(corpus: Sequence[Sequence[str]], n: int) -> Tuple[Counter, Counter]:
    """
    Build n-gram and (n-1)-gram counts with proper start/end tokens.
    """
    ngram_counts = Counter()
    context_counts = Counter()

    for sentence in corpus:
        if n == 1:
            sequence = list(sentence) + [END]
        elif n == 2:
            sequence = [START] + list(sentence) + [END]
        elif n == 3:
            sequence = [START1, START2] + list(sentence) + [END]
        else:
            sequence = [START] * (n - 1) + list(sentence) + [END]

        # Count n-grams
        for i in range(len(sequence) - n + 1):
            ngram = tuple(sequence[i:i + n])
            ngram_counts[ngram] += 1

            # Count (n-1)-gram contexts
            if n > 1:
                context = ngram[:-1]
                context_counts[context] += 1

    return ngram_counts, context_counts

### **Training the Models**
Using the frequency counts, we train two `NgramLanguageModel` instances:
- **Bigram model (n=2)** with `alpha = 1.0`
- **Trigram model (n=3)** with `alpha = 1.0`

Both models share the same vocabulary (constructed from the training set, with frequency ≥ 10), and use add-α (Laplace) smoothing.

After training, we print the **10 most frequent bigrams and trigrams** to verify that the counting and padding mechanisms behave as expected.  
Common *UNK* combinations are also observed, as they capture unseen or rare word transitions.

In [24]:
# Build counts for both models
print("Building n-gram counts...")
bigram_counts, unigram_contexts = build_ngram_counts(processed_train_corpus, 2)
trigram_counts, bigram_contexts = build_ngram_counts(processed_train_corpus, 3)

# Train models
print("Training models...")
bigram_lm = NgramLanguageModel(n=2, alpha=1.0)
bigram_lm.train_ngram(vocabulary, bigram_counts, unigram_contexts)

trigram_lm = NgramLanguageModel(n=3, alpha=1.0)
trigram_lm.train_ngram(vocabulary, trigram_counts, bigram_contexts)

print("Top 10 bigrams:")
pprint(bigram_counts.most_common(10))
print("\nTop 10 trigrams:")
pprint(trigram_counts.most_common(10))


Building n-gram counts...
Training models...
Top 10 bigrams:
[(('*UNK*', '*UNK*'), 1447),
 (('.', '*end*'), 634),
 (('the', '*UNK*'), 502),
 (('*UNK*', ','), 461),
 (('*UNK*', '.'), 418),
 (('*start*', '*UNK*'), 273),
 ((',', '*UNK*'), 251),
 (('*UNK*', 'of'), 218),
 (('*UNK*', 'to'), 181),
 (('a', '*UNK*'), 162)]

Top 10 trigrams:
[(('*UNK*', '.', '*end*'), 418),
 (('*UNK*', '*UNK*', '*UNK*'), 398),
 (('*start1*', '*start2*', '*UNK*'), 273),
 (('the', '*UNK*', '*UNK*'), 224),
 (('*UNK*', ',', '*UNK*'), 189),
 (('*UNK*', '*UNK*', '.'), 166),
 (('*start1*', '*start2*', '``'), 160),
 (('*UNK*', '*UNK*', ','), 150),
 (('*UNK*', 'the', '*UNK*'), 145),
 (('*UNK*', 'to', '*UNK*'), 110)]


### **Fine-Tuning the Smoothing Parameter (α ≤ 1)**
We then **fine-tune the Laplace smoothing parameter α** using the development (validation) corpus.  
For each model, $α$ is tested over a linear range between **0.001** and **1.0**, and the corresponding **cross-entropy** and **perplexity** values are computed.  
The value of α that minimizes the perplexity on the dev set is chosen as the **optimal α** for each model.

This process prevents under- or over-smoothing:
- Too small α → overfits frequent patterns.  
- Too large α → pushes probabilities toward uniformity.

---

In [25]:

"""## Alpha Tuning (α ≤ 1)"""

print("\n" + "="*60)
print("ALPHA PARAMETER TUNING (α ≤ 1)")
print("="*60)

# Fine-tune alpha parameters on development set (alpha <= 1)
print("Tuning Bigram Model Alpha...")
best_alpha_bigram, best_ppl_bigram = bigram_lm.fine_tune(
    processed_dev_corpus,
    alphas=np.linspace(0.001, 1.0, 20)  # Alpha from 0.001 to 1.0
)

print("\nTuning Trigram Model Alpha...")
best_alpha_trigram, best_ppl_trigram = trigram_lm.fine_tune(
    processed_dev_corpus,
    alphas=np.linspace(0.001, 1.0, 25)  # Alpha from 0.001 to 1.0
)

"""## Evaluation with Tuned Parameters"""

# Test individual probabilities with tuned alphas
print("\nTesting individual probabilities with tuned alphas:")
test_bigram = tuple(processed_dev_corpus[0][:2])
bigram_prob = bigram_lm.prob(test_bigram[1], [test_bigram[0]])
print(f"P({test_bigram[1]}|{test_bigram[0]}) = {bigram_prob:.6f} (α={bigram_lm.alpha:.3f})")

test_trigram = tuple(processed_dev_corpus[0][:3])
trigram_prob = trigram_lm.prob(test_trigram[2], test_trigram[:2])
print(f"P({test_trigram[2]}|{test_trigram[0]},{test_trigram[1]}) = {trigram_prob:.6f} (α={trigram_lm.alpha:.3f})")


ALPHA PARAMETER TUNING (α ≤ 1)
Tuning Bigram Model Alpha...
Fine-tuning 2-gram model...
Testing 20 alpha values from 0.001 to 1.0
------------------------------------------------------------
  Alpha: 0.0010, Cross Entropy: 4.0580, Perplexity: 16.6566
  Alpha: 0.0536, Cross Entropy: 3.7693, Perplexity: 13.6358
  Alpha: 0.1062, Cross Entropy: 3.7853, Perplexity: 13.7880
  Alpha: 0.1587, Cross Entropy: 3.8148, Perplexity: 14.0724
  Alpha: 0.2113, Cross Entropy: 3.8461, Perplexity: 14.3806
  Alpha: 0.2639, Cross Entropy: 3.8766, Perplexity: 14.6881
  Alpha: 0.3165, Cross Entropy: 3.9057, Perplexity: 14.9877
  Alpha: 0.3691, Cross Entropy: 3.9333, Perplexity: 15.2772
  Alpha: 0.4216, Cross Entropy: 3.9594, Perplexity: 15.5563
  Alpha: 0.4742, Cross Entropy: 3.9842, Perplexity: 15.8253
  Alpha: 0.5268, Cross Entropy: 4.0076, Perplexity: 16.0848
  Alpha: 0.5794, Cross Entropy: 4.0299, Perplexity: 16.3354
  Alpha: 0.6319, Cross Entropy: 4.0512, Perplexity: 16.5779
  Alpha: 0.6845, Cross Entro


### **Cross-Entropy and Perplexity Evaluation**
We evaluate both tuned models on the **test set**, computing:
- **Cross-Entropy (H)** – measures average uncertainty (in bits) when predicting the next token.
- **Perplexity (PPL)** – reflects the model’s effective branching factor or “average choice difficulty”.

All start tokens are excluded from the probability and token count, while `*end*` tokens are included, following the assignment’s guidelines.  
We also compare tuned models to the **default α = 1.0** baseline, reporting the relative improvement in perplexity.


In [31]:
# Cross-entropy and perplexity with tuned alphas
print("\n" + "="*60)
print("CROSS-ENTROPY AND PERPLEXITY EVALUATION (TUNED MODELS)")
print("="*60)

ce_bigram_tuned, pp_bigram_tuned = compute_cross_entropy_perplexity(bigram_lm, processed_test_corpus)
ce_trigram_tuned, pp_trigram_tuned = compute_cross_entropy_perplexity(trigram_lm, processed_test_corpus)

print(f"Bigram Model (α={bigram_lm.alpha:.3f}):")
print(f"  Cross-Entropy: {ce_bigram_tuned:.3f}")
print(f"  Perplexity:    {pp_bigram_tuned:.3f}")

print(f"\nTrigram Model (α={trigram_lm.alpha:.3f}):")
print(f"  Cross-Entropy: {ce_trigram_tuned:.3f}")
print(f"  Perplexity:    {pp_trigram_tuned:.3f}")

# Compare with default alpha=1.0
print(f"\nComparison with default alpha=1.0:")
bigram_lm_default = NgramLanguageModel(n=2, alpha=1.0)
bigram_lm_default.train_ngram(vocabulary, bigram_counts, unigram_contexts)
ce_bigram_default, pp_bigram_default = compute_cross_entropy_perplexity(bigram_lm_default, processed_test_corpus)

trigram_lm_default = NgramLanguageModel(n=3, alpha=1.0)
trigram_lm_default.train_ngram(vocabulary, trigram_counts, bigram_contexts)
ce_trigram_default, pp_trigram_default = compute_cross_entropy_perplexity(trigram_lm_default, processed_test_corpus)

print(f"Bigram:  {pp_bigram_tuned/pp_bigram_default:.2f}x {'improvement' if pp_bigram_tuned < pp_bigram_default else 'worse'}")
print(f"Trigram: {pp_trigram_tuned/pp_trigram_default:.2f}x {'improvement' if pp_trigram_tuned < pp_trigram_default else 'worse'}")


CROSS-ENTROPY AND PERPLEXITY EVALUATION (TUNED MODELS)
Bigram Model (α=0.054):
  Cross-Entropy: 3.689
  Perplexity:    12.894

Trigram Model (α=0.043):
  Cross-Entropy: 4.221
  Perplexity:    18.643

Comparison with default alpha=1.0:
Bigram:  0.77x improvement
Trigram: 0.63x improvement



### **Autocompletion Examples (Greedy and Beam Search)**
We demonstrate both decoding strategies:
- **Greedy decoding:** Selects the most probable next token at each step.
- **Beam search (width = 3):** Maintains multiple candidate continuations for more diverse completions.

Each generated sequence is cleaned of special tokens (`*start*`, `*start1*`, `*start2*`, `*end*`, `*UNK*`) for readability.  

In [30]:

"""## Autocompletion Examples with Tuned Models"""

print("\n" + "="*60)
print("AUTOCOMPLETION EXAMPLES (TUNED MODELS)")
print("="*60)

test_prefixes = [
    ["He", "has", "the"],
    ["I", "would", "like", "to", "comment", "the"],
    ["I", "was", "in"],
    ["She", "was", "not"]
]


print("GREEDY DECODING (with tuned alpha):")
print("-" * 60)

for prefix in test_prefixes:
    # Use the methods from the NgramLanguageModel class
    bigram_completion = bigram_lm.greedy_autocomplete(prefix)  # Call as method
    trigram_completion = trigram_lm.greedy_autocomplete(prefix)  # Call as method

    print(f"Prefix:  {' '.join(prefix)}")
    print(f"Bigram:  {Tools.clean_sequence(bigram_completion)} (α={bigram_lm.alpha:.3f})")
    print(f"Trigram: {Tools.clean_sequence(trigram_completion)} (α={trigram_lm.alpha:.3f})")
    print("-" * 40)

print("\nBEAM SEARCH (beam_width=3, with tuned alpha):")
print("-" * 60)

for prefix in test_prefixes:
    # Use the methods from the NgramLanguageModel class
    bigram_beam = bigram_lm.beam_search_autocomplete(prefix, beam_width=3)  # Call as method
    trigram_beam = trigram_lm.beam_search_autocomplete(prefix, beam_width=3)  # Call as method

    print(f"Prefix:  {' '.join(prefix)}")
    print(f"Bigram:  {Tools.clean_sequence(bigram_beam)} (α={bigram_lm.alpha:.3f})")
    print(f"Trigram: {Tools.clean_sequence(trigram_beam)} (α={trigram_lm.alpha:.3f})")
    print("-" * 40)

"""## Why Use Log Probabilities?"""

print("\n" + "="*60)
print("WHY USE LOG PROBABILITIES?")
print("="*60)

# Demonstration of numerical stability issues
probabilities = [0.1, 0.2, 0.15, 0.3, 0.25, 0.1, 0.05, 0.4, 0.15, 0.2]

# Raw probability product (suffers from underflow)
raw_prob = 1.0
for p in probabilities:
    raw_prob *= p

# Log probability sum (numerically stable)
log_prob_sum = sum(math.log2(p) for p in probabilities)

print(f"10-word sentence probabilities: {probabilities}")
print(f"Raw probability product: {raw_prob:.10e} (RISK: numerical underflow!)")
print(f"Log probability sum: {log_prob_sum:.3f} (STABLE: no underflow)")
print(f"Converted back: {2**log_prob_sum:.10e} (matches raw product)")

print("\nBenefits of log probabilities:")
print("1. Prevent numerical underflow with long sequences")
print("2. Convert multiplication to addition (more stable)")
print("3. Log is monotonic - preserves ranking for beam search")
print("4. Cross-entropy and perplexity naturally use logs")
print("5. Essential for reliable alpha tuning and model comparison")


AUTOCOMPLETION EXAMPLES (TUNED MODELS)
GREEDY DECODING (with tuned alpha):
------------------------------------------------------------
Prefix:  He has the
Bigram:  He has the first time . (α=0.054)
Trigram: He has the first time . (α=0.043)
----------------------------------------
Prefix:  I would like to comment the
Bigram:  I would like to comment the first time . (α=0.054)
Trigram: I would like to comment the first time . (α=0.043)
----------------------------------------
Prefix:  I was in
Bigram:  I was in the first time . (α=0.054)
Trigram: I was in there . (α=0.043)
----------------------------------------
Prefix:  She was not
Bigram:  She was not be the first time . (α=0.054)
Trigram: She was not , the one he thought with and the one he thought with and the one he thought with and the (α=0.043)
----------------------------------------

BEAM SEARCH (beam_width=3, with tuned alpha):
------------------------------------------------------------
Prefix:  He has the
Bigram:  He has 